# Exercise 2

## Imports

In [3]:
from pathlib import Path
import copy

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, models, transforms
from torchvision.transforms import v2

## Dataset Path

In [4]:
data_dir = Path("dataset")  # train/, valid/, test/

## Transforms

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    # Spatial augmentations
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),

    # Non-spatial augmentation
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
    ),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

## Datasets

In [6]:
train_dataset = datasets.ImageFolder(data_dir / "train", transform=train_transform)
valid_dataset = datasets.ImageFolder(data_dir / "valid", transform=eval_transform)
test_dataset = datasets.ImageFolder(data_dir / "test", transform=eval_transform)

n_classes = len(train_dataset.classes)

batch_size = 32
num_workers = 2

## Dataset Summary

In [7]:
print("Classes:", train_dataset.classes)
print("Train:", len(train_dataset), "Valid:", len(valid_dataset), "Test:", len(test_dataset))

Classes: ['human', 'robot']
Train: 411 Valid: 89 Test: 89


## CutMix and MixUp

In [8]:
cutmix = v2.CutMix(num_classes=n_classes, alpha=1.0)
mixup = v2.MixUp(num_classes=n_classes, alpha=0.2)

cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

## DataLoader Collate Function

In [9]:
def collate_fn(batch):
    images, labels = torch.utils.data.default_collate(batch)
    return cutmix_or_mixup(images, labels)

## DataLoaders

In [10]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [11]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)


torch.Size([16, 3, 224, 224])
torch.Size([16, 2])


## CNN Model Builder

In [12]:
def build_cnn_model(model_name, strategy="feature_extractor", num_classes=2):
    if model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        num_ftrs = model.fc.in_features
    elif model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        num_ftrs = model.classifier[1].in_features
    elif model_name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)
        num_ftrs = model.classifier[2].in_features

    if strategy == "feature_extractor" or strategy == "combined":
        for param in model.parameters():
            param.requires_grad = False

    head = nn.Linear(num_ftrs, num_classes)

    if model_name == "resnet50":
        model.fc = head
    elif model_name == "efficientnet_b0":
        model.classifier[1] = head
    elif model_name == "convnext_tiny":
        model.classifier[2] = head

    return model

## Device Setup

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


## Training Function

In [14]:
def train_model(
    model,
    train_loader,
    valid_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    num_epochs=10,
    writer=None,
):
    history = {
        "train_loss": [],
        "valid_loss": [],
        "valid_acc": [],
        "lr": [],
    }

    best_valid_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * images.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)

        model.eval()
        running_valid_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_valid_loss += loss.item() * images.size(0)

                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        epoch_valid_loss = running_valid_loss / len(valid_loader.dataset)
        epoch_valid_acc = 100.0 * correct / total
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(epoch_train_loss)
        history["valid_loss"].append(epoch_valid_loss)
        history["valid_acc"].append(epoch_valid_acc)
        history["lr"].append(current_lr)

        if epoch_valid_acc > best_valid_acc:
            best_valid_acc = epoch_valid_acc
            best_model_state = copy.deepcopy(model.state_dict())

        if writer is not None:
            epoch_number = epoch + 1
            writer.add_scalar("Loss/train", epoch_train_loss, epoch_number)
            writer.add_scalar("Loss/valid", epoch_valid_loss, epoch_number)
            writer.add_scalar("Accuracy/valid", epoch_valid_acc, epoch_number)
            writer.add_scalar("LearningRate", current_lr, epoch_number)

        if scheduler is not None:
            scheduler.step()

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"Train Loss: {epoch_train_loss:.4f} "
            f"Valid Loss: {epoch_valid_loss:.4f} "
            f"Valid Acc: {epoch_valid_acc:.2f}%"
        )

    model.load_state_dict(best_model_state)
    return history

## Evaluation Function

In [15]:
def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    avg_loss = running_loss / len(data_loader.dataset)
    accuracy = 100.0 * correct / total

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "preds": all_preds,
        "labels": all_labels,
        "probs": all_probs,
    }

## CNN Experiment Runner

In [16]:
def run_cnn_experiment(
    model_name,
    strategy="fine_tune",
    num_epochs=3,
    lr=1e-4,
    step_size=5,
    gamma=0.3,
):
    model = build_cnn_model(
        model_name=model_name,
        strategy=strategy,
        num_classes=n_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=step_size,
        gamma=gamma,
    )

    log_dir = f"tboard_logs/{model_name}_{strategy}"
    writer = SummaryWriter(log_dir=log_dir)

    all_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTraining {model_name} ({strategy})")
    print("TensorBoard log dir:", log_dir)
    print("All parameters:", all_params)
    print("Trainable parameters:", trainable_params)

    history = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=num_epochs,
        writer=writer,
    )

    test_results = evaluate_model(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
    )

    writer.close()

    return {
        "model_name": model_name,
        "strategy": strategy,
        "model": model,
        "history": history,
        "test_results": test_results,
        "all_params": all_params,
        "trainable_params": trainable_params,
        "log_dir": log_dir,
    }

## Train CNN Models

In [17]:
CNN_MODEL_NAMES = ["resnet50", "efficientnet_b0", "convnext_tiny"]
CNN_STRATEGY = "fine_tune"
CNN_NUM_EPOCHS = 5

cnn_results = {}

for model_name in CNN_MODEL_NAMES:
    cnn_results[model_name] = run_cnn_experiment(
        model_name=model_name,
        strategy=CNN_STRATEGY,
        num_epochs=CNN_NUM_EPOCHS,
    )

11.4%

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/user/dagkusue1/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100.0%



Training resnet50 (fine_tune)
TensorBoard log dir: tboard_logs/resnet50_fine_tune
All parameters: 23512130
Trainable parameters: 23512130
Epoch [1/5] Train Loss: 0.5701 Valid Loss: 0.2809 Valid Acc: 100.00%
Epoch [2/5] Train Loss: 0.3446 Valid Loss: 0.1041 Valid Acc: 98.88%
Epoch [3/5] Train Loss: 0.2479 Valid Loss: 0.1012 Valid Acc: 100.00%
Epoch [4/5] Train Loss: 0.2801 Valid Loss: 0.1066 Valid Acc: 98.88%
Epoch [5/5] Train Loss: 0.3268 Valid Loss: 0.1193 Valid Acc: 100.00%


77.6%

...


Training efficientnet_b0 (fine_tune)
TensorBoard log dir: tboard_logs/efficientnet_b0_fine_tune
All parameters: 4010110
Trainable parameters: 4010110
Epoch [1/5] Train Loss: 0.5974 Valid Loss: 0.3580 Valid Acc: 96.63%
Epoch [2/5] Train Loss: 0.4411 Valid Loss: 0.2080 Valid Acc: 97.75%
Epoch [3/5] Train Loss: 0.4138 Valid Loss: 0.1745 Valid Acc: 97.75%
Epoch [4/5] Train Loss: 0.3811 Valid Loss: 0.1454 Valid Acc: 96.63%
Epoch [5/5] Train Loss: 0.3344 Valid Loss: 0.1276 Valid Acc: 98.88%


10.1%

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /home/user/dagkusue1/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100.0%



Training convnext_tiny (fine_tune)
TensorBoard log dir: tboard_logs/convnext_tiny_fine_tune
All parameters: 27821666
Trainable parameters: 27821666
Epoch [1/5] Train Loss: 0.3848 Valid Loss: 0.0818 Valid Acc: 97.75%
Epoch [2/5] Train Loss: 0.3221 Valid Loss: 0.0824 Valid Acc: 98.88%
Epoch [3/5] Train Loss: 0.2705 Valid Loss: 0.0524 Valid Acc: 98.88%
Epoch [4/5] Train Loss: 0.2322 Valid Loss: 0.0431 Valid Acc: 100.00%
Epoch [5/5] Train Loss: 0.2219 Valid Loss: 0.0407 Valid Acc: 100.00%


## CNN Results Summary

In [18]:
cnn_summary = []

for model_name, result in cnn_results.items():
    best_valid_acc = max(result["history"]["valid_acc"])
    best_valid_epoch = result["history"]["valid_acc"].index(best_valid_acc) + 1
    test_acc = result["test_results"]["accuracy"]
    test_loss = result["test_results"]["loss"]

    cnn_summary.append({
        "model": model_name,
        "strategy": result["strategy"],
        "best_valid_acc": best_valid_acc,
        "best_valid_epoch": best_valid_epoch,
        "test_acc": test_acc,
        "test_loss": test_loss,
    })

for row in cnn_summary:
    print(
        f"{row['model']} ({row['strategy']}): "
        f"best valid acc = {row['best_valid_acc']:.2f}% "
        f"at epoch {row['best_valid_epoch']} | "
        f"test acc = {row['test_acc']:.2f}% | "
        f"test loss = {row['test_loss']:.4f}"
    )

resnet50 (fine_tune): best valid acc = 100.00% at epoch 1 | test acc = 96.63% | test loss = 0.2914
efficientnet_b0 (fine_tune): best valid acc = 98.88% at epoch 5 | test acc = 97.75% | test loss = 0.1515
convnext_tiny (fine_tune): best valid acc = 100.00% at epoch 4 | test acc = 97.75% | test loss = 0.0753


## ResNet50 Combined Strategy

In [19]:
def run_resnet50_combined_experiment(
    head_epochs=3,
    finetune_epochs=3,
    head_lr=1e-4,
    finetune_lr=1e-5,
    step_size=5,
    gamma=0.3,
):
    model = build_cnn_model(
        model_name="resnet50",
        strategy="feature_extractor",
        num_classes=n_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    all_params = sum(p.numel() for p in model.parameters())
    phase1_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("\nTraining resnet50 (combined)")
    print("Phase 1: classifier head only")
    print("All parameters:", all_params)
    print("Trainable parameters:", phase1_trainable_params)

    phase1_writer = SummaryWriter(log_dir="tboard_logs/resnet50_combined_phase1_head")
    phase1_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=head_lr,
    )
    phase1_scheduler = torch.optim.lr_scheduler.StepLR(
        phase1_optimizer,
        step_size=step_size,
        gamma=gamma,
    )

    phase1_history = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        criterion=criterion,
        optimizer=phase1_optimizer,
        scheduler=phase1_scheduler,
        device=device,
        num_epochs=head_epochs,
        writer=phase1_writer,
    )
    phase1_writer.close()

    for param in model.layer4.parameters():
        param.requires_grad = True

    phase2_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Phase 2: layer4 + classifier head")
    print("Trainable parameters:", phase2_trainable_params)

    phase2_writer = SummaryWriter(log_dir="tboard_logs/resnet50_combined_phase2_layer4")
    phase2_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=finetune_lr,
    )
    phase2_scheduler = torch.optim.lr_scheduler.StepLR(
        phase2_optimizer,
        step_size=step_size,
        gamma=gamma,
    )

    phase2_history = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        criterion=criterion,
        optimizer=phase2_optimizer,
        scheduler=phase2_scheduler,
        device=device,
        num_epochs=finetune_epochs,
        writer=phase2_writer,
    )
    phase2_writer.close()

    test_results = evaluate_model(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
    )

    return {
        "model_name": "resnet50",
        "strategy": "combined",
        "model": model,
        "phase1_history": phase1_history,
        "phase2_history": phase2_history,
        "test_results": test_results,
        "all_params": all_params,
        "phase1_trainable_params": phase1_trainable_params,
        "phase2_trainable_params": phase2_trainable_params,
    }

## Train ResNet50 Strategies

In [20]:
RESNET50_STRATEGY_EPOCHS = 5

resnet50_strategy_results = {}

resnet50_strategy_results["feature_extractor"] = run_cnn_experiment(
    model_name="resnet50",
    strategy="feature_extractor",
    num_epochs=RESNET50_STRATEGY_EPOCHS,
)

if "resnet50" in cnn_results:
    resnet50_strategy_results["fine_tune"] = cnn_results["resnet50"]
    print("Reusing fine-tuned ResNet50 result from cnn_results.")
else:
    resnet50_strategy_results["fine_tune"] = run_cnn_experiment(
        model_name="resnet50",
        strategy="fine_tune",
        num_epochs=RESNET50_STRATEGY_EPOCHS,
    )

resnet50_strategy_results["combined"] = run_resnet50_combined_experiment(
    head_epochs=RESNET50_STRATEGY_EPOCHS,
    finetune_epochs=RESNET50_STRATEGY_EPOCHS,
)


Training resnet50 (feature_extractor)
TensorBoard log dir: tboard_logs/resnet50_feature_extractor
All parameters: 23512130
Trainable parameters: 4098
Epoch [1/5] Train Loss: 0.6749 Valid Loss: 0.6264 Valid Acc: 80.90%
Epoch [2/5] Train Loss: 0.6101 Valid Loss: 0.5634 Valid Acc: 89.89%
Epoch [3/5] Train Loss: 0.5613 Valid Loss: 0.5055 Valid Acc: 95.51%
Epoch [4/5] Train Loss: 0.5357 Valid Loss: 0.4517 Valid Acc: 98.88%
Epoch [5/5] Train Loss: 0.5259 Valid Loss: 0.4275 Valid Acc: 98.88%
Reusing fine-tuned ResNet50 result from cnn_results.

Training resnet50 (combined)
Phase 1: classifier head only
All parameters: 23512130
Trainable parameters: 4098
Epoch [1/5] Train Loss: 0.6683 Valid Loss: 0.6048 Valid Acc: 84.27%
Epoch [2/5] Train Loss: 0.6167 Valid Loss: 0.5533 Valid Acc: 92.13%
Epoch [3/5] Train Loss: 0.5914 Valid Loss: 0.5076 Valid Acc: 94.38%
Epoch [4/5] Train Loss: 0.5428 Valid Loss: 0.4688 Valid Acc: 94.38%
Epoch [5/5] Train Loss: 0.5275 Valid Loss: 0.4329 Valid Acc: 95.51%
Phas

## ResNet50 Strategy Summary

In [21]:
resnet50_strategy_summary = []

for strategy, result in resnet50_strategy_results.items():
    if strategy == "combined":
        valid_accs = result["phase1_history"]["valid_acc"] + result["phase2_history"]["valid_acc"]
        best_valid_acc = max(valid_accs)
        best_valid_epoch = valid_accs.index(best_valid_acc) + 1
    else:
        best_valid_acc = max(result["history"]["valid_acc"])
        best_valid_epoch = result["history"]["valid_acc"].index(best_valid_acc) + 1

    resnet50_strategy_summary.append({
        "strategy": strategy,
        "best_valid_acc": best_valid_acc,
        "best_valid_epoch": best_valid_epoch,
        "test_acc": result["test_results"]["accuracy"],
        "test_loss": result["test_results"]["loss"],
    })

for row in resnet50_strategy_summary:
    print(
        f"resnet50 ({row['strategy']}): "
        f"best valid acc = {row['best_valid_acc']:.2f}% "
        f"at epoch {row['best_valid_epoch']} | "
        f"test acc = {row['test_acc']:.2f}% | "
        f"test loss = {row['test_loss']:.4f}"
    )

resnet50 (feature_extractor): best valid acc = 98.88% at epoch 4 | test acc = 95.51% | test loss = 0.4680
resnet50 (fine_tune): best valid acc = 100.00% at epoch 1 | test acc = 96.63% | test loss = 0.2914
resnet50 (combined): best valid acc = 97.75% at epoch 8 | test acc = 96.63% | test loss = 0.3328


## DINOv2 Classifier

In [22]:
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")

        for param in self.backbone.parameters():
            param.requires_grad = False

        self.head = nn.Linear(384, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

## DINOv2 Experiment Runner

In [23]:
def run_dinov2_experiment(
    num_epochs=3,
    lr=1e-4,
    step_size=5,
    gamma=0.3,
):
    model = DINOv2Classifier(num_classes=n_classes).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=step_size,
        gamma=gamma,
    )

    log_dir = "tboard_logs/dinov2_feature_extractor"
    writer = SummaryWriter(log_dir=log_dir)

    all_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("\nTraining DINOv2 (feature_extractor)")
    print("TensorBoard log dir:", log_dir)
    print("All parameters:", all_params)
    print("Trainable parameters:", trainable_params)

    history = train_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=num_epochs,
        writer=writer,
    )

    test_results = evaluate_model(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
    )

    writer.close()

    return {
        "model_name": "dinov2_vits14",
        "strategy": "feature_extractor",
        "model": model,
        "history": history,
        "test_results": test_results,
        "all_params": all_params,
        "trainable_params": trainable_params,
        "log_dir": log_dir,
    }

## Train DINOv2

In [24]:
DINO_NUM_EPOCHS = 5

dinov2_result = run_dinov2_experiment(
    num_epochs=DINO_NUM_EPOCHS,
)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /home/user/dagkusue1/.cache/torch/hub/main.zip


/home/user/dagkusue1/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/user/dagkusue1/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/user/dagkusue1/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
10.4%

Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /home/user/dagkusue1/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100.0%



Training DINOv2 (feature_extractor)
TensorBoard log dir: tboard_logs/dinov2_feature_extractor
All parameters: 22057346
Trainable parameters: 770
Epoch [1/5] Train Loss: 0.9296 Valid Loss: 0.8399 Valid Acc: 65.17%
Epoch [2/5] Train Loss: 0.7554 Valid Loss: 0.5350 Valid Acc: 76.40%
Epoch [3/5] Train Loss: 0.5867 Valid Loss: 0.3889 Valid Acc: 83.15%
Epoch [4/5] Train Loss: 0.5658 Valid Loss: 0.3073 Valid Acc: 88.76%
Epoch [5/5] Train Loss: 0.4737 Valid Loss: 0.2500 Valid Acc: 93.26%


## DINOv2 Results Summary

In [25]:
best_valid_acc = max(dinov2_result["history"]["valid_acc"])
best_valid_epoch = dinov2_result["history"]["valid_acc"].index(best_valid_acc) + 1
test_acc = dinov2_result["test_results"]["accuracy"]
test_loss = dinov2_result["test_results"]["loss"]

print(
    f"DINOv2 ({dinov2_result['strategy']}): "
    f"best valid acc = {best_valid_acc:.2f}% "
    f"at epoch {best_valid_epoch} | "
    f"test acc = {test_acc:.2f}% | "
    f"test loss = {test_loss:.4f}"
)

DINOv2 (feature_extractor): best valid acc = 93.26% at epoch 5 | test acc = 84.27% | test loss = 0.3607
